In [1]:
pip install flourish

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 4.1 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 6.0 MB/s  0:00:00 eta 0:00:01
  Created wheel for feedgen: filename=feedgen-1.0.0-py2.py3-none-any.whl size=45393 sha256=7dc891033c76ec2b1281b8a7fcd1c969c6c08e254e55c04521329e27433d9b70
  Stored in directory: /Users/josemessiasferreira/Library/Caches/pip/wheels/0b/c9/59/aca2e0200bb16ebe379190a250155dfbab4073c7c7c7fb2821
Successfully built feedgen
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [flourish]/11 [flourish]r]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
import urllib3
import pandas as pd
import io

# Desactivar avisos de seguridad (por el problema de SSL que tuvimos)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- CONFIGURACIÓN ---
FLOURISH_IP = "34.120.208.157"
VIZ_ID = "28079074" 
API_KEY = "TU_API_KEY_AQUÍ" # <--- Pon aquí tu clave de Flourish

# URL de la API usando la IP que nos funcionó
# Intenta con esta estructura que es la oficial para actualizar datos:
url = f"https://{FLOURISH_IP}/api/v1/visualisations/{VIZ_ID}/data"
# --- DATOS DE PRUEBA (RH) ---
# Aquí puedes cargar tu propio Excel: df = pd.read_excel("empleados.xlsx")
df = pd.DataFrame({
    "Departamento": ["IT", "Ventas", "Marketing", "Finanzas"],
    "Sueldo Medio": [45000, 38000, 35000, 42000],
    "Nº Empleados": [12, 25, 8, 5]
})

# Preparar CSV en memoria
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, index=False)
csv_content = csv_buffer.getvalue()

headers = {"Host": "api.flourish.studio"}

try:
    print(f"Enviando datos a la visualización {VIZ_ID}...")
    
    response = requests.post(
        url,
        params={"api_key": API_KEY},
        headers=headers,
        files={"data": ("data.csv", csv_content)},
        verify=False,
        timeout=20
    )
    
    if response.status_code == 200:
        print("✅ ¡Éxito! Los datos han sido actualizados.")
        print(f"Puedes ver los cambios aquí: https://flourish.studio/visualisation/{VIZ_ID}/")
    elif response.status_code == 401:
        print("❌ Error 401: Tu API Key no es válida o tu cuenta no tiene permisos de API.")
    else:
        print(f"⚠️ Error {response.status_code}: {response.text}")

except Exception as e:
    print(f"🚨 Fallo en la conexión: {e}")

Enviando datos a la visualización 28079074...
⚠️ Error 404: <!DOCTYPE html><html><head><meta charSet="utf-8"/><meta http-equiv="x-ua-compatible" content="ie=edge"/><meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no"/><meta name="generator" content="Gatsby 4.25.9"/><meta name="theme-color" content="#208C27"/><style data-href="/styles.f2213270e726f34cdecb.css" data-identity="gatsby-global-css">.yarl__slide_captions_container{background:var(--yarl__slide_captions_container_background,rgba(0,0,0,.5));left:var(--yarl__slide_captions_container_left,0);padding:var(--yarl__slide_captions_container_padding,16px);position:absolute;right:var(--yarl__slide_captions_container_right,0);-webkit-transform:translateZ(0)}.yarl__slide_title{color:var(--yarl__slide_title_color,#fff);font-size:var(--yarl__slide_title_font_size,125%);font-weight:var(--yarl__slide_title_font_weight,bolder);max-width:calc(100% - var(--yarl__toolbar_width, 0px));overflow:hidden;text-overflow:e

In [2]:
import requests
import urllib3
import io

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuración
FLOURISH_IP = "34.120.208.157"
VIZ_ID = "28079074"
API_KEY = "TU_API_KEY_REAL" # <--- ¡Cámbiala!

# Estructura oficial de la API de Flourish
url = f"https://{FLOURISH_IP}/api/v1/visualisations/{VIZ_ID}/data"

# Datos de prueba sencillos
csv_content = "Departamento,Empleados\nIT,15\nVentas,30\nRH,5"

headers = {
    "Host": "api.flourish.studio",
    "Content-Type": "application/json" # A veces la API prefiere saber el tipo
}

try:
    print(f"Enviando a: {url}")
    
    # En la API de Flourish, a veces los datos se envían como JSON 
    # que contiene el string del CSV, o como un archivo. 
    # Probemos primero como archivo (Multipart):
    
    response = requests.post(
        url,
        params={"api_key": API_KEY},
        headers={"Host": "api.flourish.studio"},
        files={"data": ("data.csv", csv_content)},
        verify=False
    )
    
    if response.status_code == 200:
        print("✅ ¡LOGRADO! Datos actualizados.")
    else:
        print(f"❌ Error {response.status_code}")
        # Si recibes HTML en la respuesta, es que la IP no es la correcta para la API
        if "<!DOCTYPE html>" in response.text:
             print("Aviso: El servidor devolvió una página web, no una respuesta de API.")
             print("Esto confirma que la IP directa no está llegando al motor de la API.")

except Exception as e:
    print(f"🚨 Error: {e}")

Enviando a: https://34.120.208.157/api/v1/visualisations/28079074/data
❌ Error 404
Aviso: El servidor devolvió una página web, no una respuesta de API.
Esto confirma que la IP directa no está llegando al motor de la API.


In [3]:
import pandas as pd
import os

# 1. Carga tus datos reales de RH (Excel o CSV)
# df = pd.read_excel("tu_archivo_de_rh.xlsx")

# 2. Creemos unos datos de ejemplo que suelen usarse en RH
data = {
    'Departamento': ['IT', 'Ventas', 'Marketing', 'Operaciones', 'Finanzas'],
    'Empleados': [15, 28, 12, 40, 8],
    'Sueldo_Promedio': [4500, 3200, 3500, 2800, 4100],
    'Rotacion_%': [5, 15, 10, 20, 2]
}

df = pd.DataFrame(data)

# 3. Guardar el archivo en tu escritorio para subirlo fácil
escritorio = os.path.expanduser("~/Desktop/datos_listos_flourish.csv")
df.to_csv(escritorio, index=False)

print(f"✅ ¡Archivo generado con éxito!")
print(f"📍 Búscalo en tu escritorio como: datos_listos_flourish.csv")
print(f"👉 Ahora ve a https://app.flourish.studio/visualisation/28079074/edit")
print(f"👉 Haz clic en 'Data' -> 'Upload data' y selecciona este archivo.")

✅ ¡Archivo generado con éxito!
📍 Búscalo en tu escritorio como: datos_listos_flourish.csv
👉 Ahora ve a https://app.flourish.studio/visualisation/28079074/edit
👉 Haz clic en 'Data' -> 'Upload data' y selecciona este archivo.
